# ASD Detection from Resting-State fMRI: Multi-Atlas Connectivity + Deep Learning Pipeline

This notebook builds and evaluates a classifier that distinguishes individuals with Autism Spectrum Disorder (ASD) from typically developing controls, using resting-state fMRI data from the ABIDE dataset (C-PAC preprocessing). It combines multi-atlas tangent connectivity features with a compact neural network, and is designed around leakage-free, reproducible evaluation.

| # | Design feature | How it is implemented |
|---|---|---|
| 1 | Single, clear pipeline | One training/evaluation function handles seeding, scaling, PCA, training, threshold selection, and testing |
| 2 | Leakage-free threshold selection | The classification threshold is tuned on a validation split carved from the training fold only, never on test labels |
| 3 | Secure data download | SSL verification stays **on**; a documented, opt-in fallback exists for restricted environments |
| 4 | Untouched final test set | A stratified **hold-out set** is split off before cross-validation and evaluated exactly once |
| 5 | Per-fold reproducibility | Every fold seeds NumPy and PyTorch independently |
| 6 | Saved model artifacts | Scaler, PCA, model weights, and metadata are saved to disk, ready for upload to S3 |
| 7 | Interpretability | Permutation feature importance is computed on the final model |
| 8 | Fast re-runs | Connectivity features are cached to disk after the first computation |

**Pipeline overview:** load ABIDE → extract time-series from two atlases (Schaefer-2018 and AAL) → compute tangent connectivity per atlas → concatenate features → cross-validate → train final model → evaluate on hold-out set → save artifacts → interpret.

Run the cells top to bottom.

## Step 1 — Setup
Install and import everything needed.

In [ ]:
!pip install -q nilearn scikit-learn pandas numpy torch tqdm joblib

import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import joblib

from nilearn.datasets import fetch_abide_pcp, fetch_atlas_schaefer_2018, fetch_atlas_aal
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator, ClassifierMixin

from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm


## Step 2 — Config & reproducibility helper

`set_seed()` is called **once globally** and again **inside every fold**, so each fold's
weight initialization and data shuffling is reproducible on its own, not just the first one.


In [ ]:
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ARTIFACT_DIR = "artifacts"
CACHE_DIR = "feature_cache"
os.makedirs(ARTIFACT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("Device:", DEVICE)


## Step 3 — Data loading

Same as the original: fetch 200 subjects from ABIDE (C-PAC preprocessing pipeline).
Increase `N_SUBJECTS` later for a more generalizable model — the original used only
~18% of the full ABIDE release.


In [ ]:
N_SUBJECTS = 200

print("Fetching ABIDE...")
abide = fetch_abide_pcp(n_subjects=N_SUBJECTS, pipeline="cpac")

phenotypic = pd.DataFrame(abide.phenotypic)
y = (phenotypic["DX_GROUP"] == 1).astype(int).values
print(f"Loaded {len(y)} subjects | ASD={y.sum()} | Control={len(y) - y.sum()}")


## Step 4 — Multi-atlas feature extraction (SSL fix + caching)

**Fix #3 (security):** SSL verification is left **on** by default. If an atlas download
fails in your environment with a certificate error, the real fix is almost always to
update the `certifi` package (`pip install --upgrade certifi`) rather than disabling
verification. An explicit, opt-in `ALLOW_INSECURE_SSL` flag is provided below purely as
a last resort — keep it `False` unless you understand the risk.

**Fix #8 (speed/cost):** the extracted tangent-connectivity features are cached to disk
(`feature_cache/features.npz`). Re-running the notebook won't re-download or
re-extract 200 subjects' worth of fMRI data every time — useful locally and essential
on AWS, where you'd cache this straight to S3.


In [ ]:
ALLOW_INSECURE_SSL = False  # keep False; see markdown above

CACHE_PATH = os.path.join(CACHE_DIR, "features.npz")

if os.path.exists(CACHE_PATH):
    print("Loading cached connectivity features...")
    cached = np.load(CACHE_PATH)
    X = cached["X"]
else:
    def _fetch_atlases():
        atlas1 = fetch_atlas_schaefer_2018(n_rois=100)
        atlas2 = fetch_atlas_aal()
        return atlas1, atlas2

    if ALLOW_INSECURE_SSL:
        import requests
        from unittest.mock import patch
        original_send = requests.Session.send

        def _insecure_send(self, request, **kwargs):
            kwargs["verify"] = False
            return original_send(self, request, **kwargs)

        print("WARNING: fetching atlases with SSL verification disabled.")
        with patch("requests.Session.send", new=_insecure_send):
            atlas1, atlas2 = _fetch_atlases()
    else:
        atlas1, atlas2 = _fetch_atlases()

    masker1 = NiftiLabelsMasker(labels_img=atlas1.maps, standardize="zscore_sample")
    masker2 = NiftiLabelsMasker(labels_img=atlas2.maps, standardize="zscore_sample")

    ts1, ts2 = [], []
    print("Extracting time-series from multi-atlas...")
    for img in tqdm(abide.func_preproc):
        ts1.append(masker1.fit_transform(img))
        ts2.append(masker2.fit_transform(img))

    conn1 = ConnectivityMeasure(kind="tangent")
    conn2 = ConnectivityMeasure(kind="tangent")
    print("Computing connectivity measures...")
    X1 = conn1.fit_transform(ts1)
    X2 = conn2.fit_transform(ts2)

    X1 = np.array([c.flatten() for c in X1])
    X2 = np.array([c.flatten() for c in X2])

    X = np.concatenate([X1, X2], axis=1)
    np.savez_compressed(CACHE_PATH, X=X)
    print("Cached features to", CACHE_PATH)

print(f"Feature matrix shape: {X.shape}")


## Step 5 — Model definition

Unchanged from the original — a compact MLP with LayerNorm + Dropout for regularization.


In [ ]:
class ASDModel(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU(),

            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x)


## Step 6 — Training & leakage-free threshold tuning

**Fix #2 (the critical one):** `pick_threshold()` is only ever called on a validation
split carved out of the *training* data. The test set's labels are never used for
anything except the final accuracy/AUC computation.

`train_one_fold()` wraps: seed -> scale -> (optional) PCA -> train -> tune threshold on
validation -> evaluate on test. This single function replaces both duplicated loops
from the original notebook.


In [ ]:
def pick_threshold(probs_val: np.ndarray, y_val: np.ndarray) -> float:
    """Youden's J statistic threshold, computed ONLY on a validation split."""
    fpr, tpr, thresholds = roc_curve(y_val, probs_val)
    best_idx = np.argmax(tpr - fpr)
    return float(thresholds[best_idx])


def train_one_fold(X_train_full, y_train_full, X_test, y_test, fold_seed, use_pca=True, n_components=50,
                    epochs=40, batch_size=16, lr=3e-4, weight_decay=1e-4, val_size=0.15):
    set_seed(fold_seed)

    # Carve out an internal validation split from the TRAIN data only (test stays untouched)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_full, y_train_full, test_size=val_size, stratify=y_train_full, random_state=fold_seed
    )

    # Scale: fit on X_tr only
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    # Optional PCA: fit on X_tr only
    pca = None
    if use_pca:
        pca = PCA(n_components=n_components, random_state=fold_seed)
        X_tr_s = pca.fit_transform(X_tr_s)
        X_val_s = pca.transform(X_val_s)
        X_test_s = pca.transform(X_test_s)

    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr_s), torch.FloatTensor(y_tr).view(-1, 1)),
        batch_size=batch_size, shuffle=True,
    )

    model = ASDModel(X_tr_s.shape[1]).to(DEVICE)

    pos_weight = torch.tensor([(len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1)], dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    model.train()
    for _ in range(epochs):
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            loss = criterion(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        val_probs = torch.sigmoid(model(torch.FloatTensor(X_val_s).to(DEVICE))).cpu().numpy().ravel()
        test_probs = torch.sigmoid(model(torch.FloatTensor(X_test_s).to(DEVICE))).cpu().numpy().ravel()

    # Threshold chosen using VALIDATION data, applied to TEST data
    threshold = pick_threshold(val_probs, y_val)
    test_preds = (test_probs > threshold).astype(int)

    acc = accuracy_score(y_test, test_preds)
    auc = roc_auc_score(y_test, test_probs)

    return {
        "accuracy": acc, "auc": auc, "threshold": threshold,
        "model": model, "scaler": scaler, "pca": pca,
    }


## Step 7 — Hold-out test set + cross-validation

**Fix #4:** a stratified hold-out test set (15% of subjects) is split off **before**
anything else and is never touched during cross-validation. Cross-validation runs on
the remaining 85% purely to report a robust, variance-aware estimate of performance.
The final model is then retrained on the full CV portion and evaluated exactly once
on the untouched hold-out set — that number is the one to actually report/publish.


In [ ]:
X_cv, X_holdout, y_cv, y_holdout = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)
print(f"CV set: {X_cv.shape[0]} subjects | Hold-out set: {X_holdout.shape[0]} subjects")

USE_PCA = True  # flip to False to reproduce the original notebook's "no-PCA" variant

kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
fold_results = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X_cv, y_cv)):
    print(f"\n--- Fold {fold + 1} ---")
    result = train_one_fold(
        X_cv[train_idx], y_cv[train_idx], X_cv[test_idx], y_cv[test_idx],
        fold_seed=SEED + fold, use_pca=USE_PCA,
    )
    fold_results.append(result)
    print(f"Accuracy: {result['accuracy']:.3f} | AUC: {result['auc']:.3f} | threshold: {result['threshold']:.3f}")

cv_acc = [r["accuracy"] for r in fold_results]
cv_auc = [r["auc"] for r in fold_results]

print("\n===== CROSS-VALIDATION RESULTS (model-selection estimate) =====")
print(f"Mean Accuracy: {np.mean(cv_acc) * 100:.2f}% (+/- {np.std(cv_acc) * 100:.2f})")
print(f"Mean AUC:      {np.mean(cv_auc):.3f} (+/- {np.std(cv_auc):.3f})")


## Step 8 — Final model: train on all CV data, evaluate once on the hold-out set

This is the number that should go in a report — it has never been seen by the model
or used to pick a threshold or hyperparameter.


In [ ]:
final = train_one_fold(X_cv, y_cv, X_holdout, y_holdout, fold_seed=SEED, use_pca=USE_PCA)

print("===== FINAL HOLD-OUT RESULTS =====")
print(f"Accuracy: {final['accuracy'] * 100:.2f}%")
print(f"AUC:      {final['auc']:.3f}")
print(f"Threshold used: {final['threshold']:.3f}")


## Step 9 — Save artifacts

**Fix #6:** the scaler, PCA, and model weights are saved so the pipeline is actually
deployable — e.g. upload this `artifacts/` folder to S3 and load it inside a
SageMaker inference endpoint.


In [ ]:
torch.save(final["model"].state_dict(), os.path.join(ARTIFACT_DIR, "model.pt"))
joblib.dump(final["scaler"], os.path.join(ARTIFACT_DIR, "scaler.joblib"))
if final["pca"] is not None:
    joblib.dump(final["pca"], os.path.join(ARTIFACT_DIR, "pca.joblib"))

with open(os.path.join(ARTIFACT_DIR, "metadata.json"), "w") as f:
    json.dump({
        "input_dim": X.shape[1],
        "use_pca": USE_PCA,
        "threshold": final["threshold"],
        "cv_mean_accuracy": float(np.mean(cv_acc)),
        "cv_mean_auc": float(np.mean(cv_auc)),
        "holdout_accuracy": float(final["accuracy"]),
        "holdout_auc": float(final["auc"]),
        "seed": SEED,
    }, f, indent=2)

print("Saved model, scaler, PCA and metadata to:", ARTIFACT_DIR)


## Step 10 — Interpretability: permutation feature importance

**Fix #7:** rather than treating the MLP as a black box, this wraps the trained model
in a scikit-learn-compatible class and uses `permutation_importance` on the hold-out
set to see which (PCA-transformed) features actually move predictions. For a clinical
project, the next step would be projecting important PCA components back onto
original connectivity edges / brain regions.


In [ ]:
class TorchModelWrapper(ClassifierMixin, BaseEstimator):  # FIX: ClassifierMixin FIRST
    def __init__(self, model, device, threshold=0.5):
        self.model = model
        self.device = device
        self.threshold = threshold
        self.classes_ = np.array([0, 1])

    def fit(self, X, y):
        return self  # already trained

    def predict_proba(self, X):
        self.model.eval()
        with torch.no_grad():
            probs = torch.sigmoid(
                self.model(torch.FloatTensor(X).to(self.device))
            ).cpu().numpy().ravel()
        return np.column_stack([1 - probs, probs])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > self.threshold).astype(int)


# Rebuild the exact preprocessing used for the hold-out set
X_holdout_s = final["scaler"].transform(X_holdout)
if final["pca"] is not None:
    X_holdout_s = final["pca"].transform(X_holdout_s)

wrapper = TorchModelWrapper(final["model"], DEVICE, threshold=final["threshold"])
importance = permutation_importance(
    wrapper, X_holdout_s, y_holdout, scoring="roc_auc", n_repeats=20, random_state=SEED
)

top_k = 10
order = np.argsort(importance.importances_mean)[::-1][:top_k]
print(f"Top {top_k} most important components (by drop in AUC when shuffled):")
for rank, idx in enumerate(order, 1):
    comp_name = f"PCA component {idx}" if USE_PCA else f"raw feature {idx}"
    print(f"{rank}. {comp_name}: {importance.importances_mean[idx]:.4f} (+/- {importance.importances_std[idx]:.4f})")

## Summary of the Pipeline

This notebook builds and evaluates an ASD classifier from resting-state fMRI data (ABIDE, C-PAC preprocessing) using multi-atlas functional connectivity and a compact neural network.

**Feature extraction**
- Time-series are extracted from two atlases (Schaefer-2018 with 100 regions, and AAL).
- A separate tangent-space connectivity estimator is fitted for each atlas, since the atlases have different region counts.
- The two connectivity matrices are flattened and concatenated into one feature vector per subject.
- Features are cached to disk (`feature_cache/features.npz`), so re-runs skip the slow extraction step.

**Training and evaluation**
- One single training function handles seeding, scaling, optional PCA, training, threshold selection, and evaluation.
- Scaler and PCA are fitted on training data only, so no information leaks from test subjects.
- The classification threshold (Youden's J) is tuned on a validation split taken from the training data, never on test labels.
- A stratified 15% hold-out set is separated before cross-validation and is evaluated exactly once, so its result is the headline number.
- 10-fold stratified cross-validation gives a variance-aware performance estimate.
- Every fold is seeded independently for reproducibility.

**Model**
- A compact MLP (Linear → LayerNorm → ReLU → Dropout → Linear → LayerNorm → ReLU → Linear), trained with Adam and a class-weighted binary cross-entropy loss to handle ASD/control imbalance.

**Deployment and interpretability**
- The trained model, scaler, PCA, and run metadata are saved to `artifacts/`, ready to upload to S3 or serve from an inference endpoint.
- Permutation importance, using a scikit-learn-compatible wrapper around the network, ranks the components that most affect predictions.
- SSL verification stays on during atlas downloads; disabling it is an explicit opt-in that is off by default.

**Next steps:** increase `N_SUBJECTS` toward ABIDE's full ~1,100 subjects, add leave-one-site-out and ComBat-harmonized evaluation to test robustness to scanner-site effects, and use nested cross-validation to tune `n_components` and the learning rate.